# D-10 · Examen ciego único · Solo Bruno
Claude sella 10–20 PDFs propios de `Drive/d10-examen` publicando solamente SHA-256,
número de páginas y tipo. Codex no recibe los PDFs. Este cuaderno NO entrena:
requiere las dos etapas ya congeladas (20 épocas sintéticas y 5 de afinado A).

Completar la configuración con hashes conservados ANTES de abrir el examen.
Los PDFs se abren una vez por el evaluador; no se escriben texto, recortes,
fuentes, imágenes ni temporales de páginas. Solo métricas agregadas y control
de integridad. No ejecutar desde dos runtimes simultáneos ni borrar el marcador.
Un fallo tras iniciar también consume el intento. No hay reintento automático.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os, sys, subprocess, json, getpass, base64, hashlib, re
os.nice(max(0, 15-os.getpriority(os.PRIO_PROCESS, 0)))
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
REPO = 'https://github.com/brune6320-del/prometeo-studio.git'
BRANCH = 'studio/adn-tipografico'
# Código verificado antes del sello; no usar el HEAD futuro sin nueva revisión.
CODE_COMMIT = '36ee1f29c86d58db01ee6bdaf2a2f0dc72e35df1'
PRERREGISTRO_SHA = '16b5a933c466c04eaa9a28e06dcb62ceb4be2c66b73e1391b1a16d546637d776'
CHECKOUT = Path('/content/prometeo-adn-examen')
assert not CHECKOUT.exists(), 'Usar un runtime nuevo; no sustituir un checkout existente.'
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = getpass.getpass('Token GitHub de lectura (no se guarda): ')
assert token, 'Se necesita acceso de lectura al repo privado'
auth = base64.b64encode(('x-access-token:'+token).encode()).decode()
env = os.environ.copy()
env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.extraHeader',
           GIT_CONFIG_VALUE_0='Authorization: Basic '+auth)
subprocess.run(['git','clone','--branch',BRANCH,REPO,str(CHECKOUT)],env=env,check=True)
del token, auth, env
subprocess.run(['git','checkout','--detach',CODE_COMMIT],cwd=CHECKOUT,check=True)
CODE = CHECKOUT/'tools/adn_tipo'
assert hashlib.sha256((CODE/'PRERREGISTRO.md').read_bytes()).hexdigest() == PRERREGISTRO_SHA
print('Código:', CODE_COMMIT)
print('Prerregistro:', PRERREGISTRO_SHA)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements.txt')],check=True)
# Se monta Drive solo después de verificar código y protocolo.
drive.mount('/content/drive')


## Configuración sin abrir el examen
Claude aporta `SELLO_SHA`. El entrenamiento aporta los otros tres SHA y las rutas
al catálogo y checkpoints (cada checkpoint junto a su `entrenamiento.json`).
No calcular de nuevo estos SHA desde archivos cambiados para hacer pasar una
verificación. El sello se puede guardar fuera de `d10-examen`.
La carpeta de control debe ser siempre la misma para todos los intentos.


In [ ]:
CARPETA = Path('/content/drive/MyDrive/d10-examen')
SELLO = Path('/content/drive/MyDrive/examen-sellado.json')
SELLO_SHA = None  # SHA publicado por Claude, de los bytes exactos del JSON
GALERIA = None  # carpeta sintética congelada que contiene galeria.json
GALERIA_SHA = None
PESOS_SINTETICO = None  # .../solo-sintetico/pesos.pt; 20 épocas
PESOS_SINTETICO_SHA = None
PESOS_AFINADO = None  # .../afinado-A/pesos.pt; 5 épocas, mismos pesos de partida
PESOS_AFINADO_SHA = None
CONTROL = Path('/content/drive/MyDrive/Prometeo/ADN/control-examen')
SALIDA = Path('/content/drive/MyDrive/Prometeo/ADN/examen-real-metricas.json')
DISPOSITIVO = 'cuda'  # 'cpu' permitido para inferencia, no reentrena
COSTE_HORA_USD = None  # tarifa GPU efectiva de Bruno; None = desconocido
for h in (SELLO_SHA, GALERIA_SHA, PESOS_SINTETICO_SHA, PESOS_AFINADO_SHA):
    assert isinstance(h,str) and re.fullmatch('[0-9a-f]{64}',h), 'Completar hashes previamente conservados'
assert all(p is not None for p in (GALERIA, PESOS_SINTETICO, PESOS_AFINADO)), 'Completar rutas de modelos/catalogo'
# Ninguna celda anterior lee, lista o analiza los PDF de examen.


## Única evaluación
Ejecutar esta celda una sola vez. El control persiste en Drive, no en el runtime.
Verifica carpeta exacta, SHA, procedencia, páginas y tipos; mide las tres filas
en una pasada. No vuelve a entrenar ni selecciona modelos por el resultado.
Los escaneados sin nativo emparejado quedan NO EVALUABLE con métricas null.


In [ ]:
cmd = [sys.executable, str(CODE/'medir_examen.py'), str(CARPETA), str(SELLO),
    '--sello-sha256', SELLO_SHA, '--prerregistro-sha256', PRERREGISTRO_SHA,
    '--galeria', str(GALERIA), '--galeria-sha256', GALERIA_SHA,
    '--sintetico', str(PESOS_SINTETICO), '--sintetico-sha256', PESOS_SINTETICO_SHA,
    '--afinado', str(PESOS_AFINADO), '--afinado-sha256', PESOS_AFINADO_SHA,
    '--control', str(CONTROL), '--salida', str(SALIDA), '--dispositivo', DISPOSITIVO,
    '--propios-bruno-sin-firmas-ni-sellos']
if COSTE_HORA_USD is not None:
    cmd += ['--coste-hora-usd', str(COSTE_HORA_USD)]
# No imprimir cmd, stderr o traceback con rutas. El evaluador emite JSON sanitizado.
result = subprocess.run(cmd, cwd=CODE, capture_output=True, text=True)
try:
    agregado = json.loads(result.stdout)
except (ValueError, TypeError):
    agregado = {'estado':'RECHAZADO','codigo':'EVALUADOR_NO_DEVOLVIO_JSON'}
print(json.dumps(agregado, ensure_ascii=False, indent=2))
if result.returncode != 0:
    print('Examen rechazado. No repetir si el control ya marca INICIADO o FALLIDO_NO_REPETIR.')
else:
    print('Solo métricas agregadas guardadas; conservar el control y no volver a ejecutar.')
del result, cmd


Compartir solamente el JSON agregado. No compartir PDFs, recortes, ADN o texto.
La métrica corresponde al subconjunto de familias/glifos elegibles con segmentación
y transcripción conocidas por A; revisar cobertura y exclusiones. No confundir
NO EVALUABLE con un 0% medido. Si el alumno no supera al vecino, es un resultado
negativo que se conserva. No usarlo para volver a elegir pesos o umbrales.
